In [ ]:
#!/usr/bin/env python3
"""
mlflow_diagnose.py — Diagnóstico end-to-end para MLflow Tracking + Artifacts.

Uso:
  export TRACKING_URI=http://172.16.0.200:5000
  python mlflow_diagnose.py --model-name m-c940010728064a7db2ec089e0b552ae6 --stage latest
  # o con versión exacta:
  python mlflow_diagnose.py --model-name m-c940010728064a7db2ec089e0b552ae6 --version 7
"""

import argparse
import json
import os
import sys
import tempfile
import time
import traceback
from urllib.parse import urlparse, urlunparse

import mlflow
from mlflow import MlflowClient

try:
    import requests
except Exception:
    requests = None  # requests es opcional para las pruebas HTTP directas

def short_exc(e: Exception) -> str:
    return f"{type(e).__name__}: {e}"

def maybe_http_check(tracking_uri: str):
    if not requests:
        print("[WARN] 'requests' no está instalado; omito pruebas HTTP directas.")
        return

    try:
        u = urlparse(tracking_uri)
        if not u.scheme or not u.netloc:
            print(f"[WARN] TRACKING_URI no parece URL HTTP válida: {tracking_uri}")
            return

        base = urlunparse((u.scheme, u.netloc, "", "", "", ""))
        # Endpoints clave:
        urls = [
            ("/api/2.0/mlflow/experiments/list", "experiments/list"),
            ("/api/2.0/mlflow/runs/search", "runs/search"),
            ("/api/2.0/mlflow-artifacts/health", "artifacts/health"),
        ]
        for path, label in urls:
            url = f"{base}{path}"
            try:
                r = requests.get(url, timeout=8)
                print(f"[HTTP] GET {label}: {r.status_code}")
                if r.status_code >= 400:
                    print(f"       Body: {r.text[:400]}...")
            except Exception as e:
                print(f"[HTTP][ERROR] {label}: {short_exc(e)}")
    except Exception as e:
        print(f"[HTTP][ERROR] chequeos básicos: {short_exc(e)}")

def list_metadata(client: MlflowClient):
    # Experiments
    try:
        exps = client.search_experiments()
        print(f"[TRACKING] Experiments encontrados: {len(exps)}")
        for e in exps[:10]:
            print(f"  - {e.experiment_id}: {e.name} (lifecycle={e.lifecycle_stage})")
        if len(exps) > 10:
            print("  ...")
    except Exception as e:
        print(f"[TRACKING][ERROR] listar experiments: {short_exc(e)}")

    # Registered models
    try:
        rms = client.search_registered_models()
        print(f"[REGISTRY] Registered models: {len(rms)}")
        for rm in rms[:10]:
            name = rm.name
            print(f"  - {name}")
            try:
                latest = client.get_latest_versions(name)
                for v in latest:
                    print(f"      version={v.version} stage={v.current_stage} run_id={v.run_id}")
            except Exception as ie:
                print(f"      [WARN] No pude obtener latest versions: {short_exc(ie)}")
        if len(rms) > 10:
            print("  ...")
    except Exception as e:
        print(f"[REGISTRY][ERROR] listar registered models: {short_exc(e)}")

def resolve_model_version(client: MlflowClient, model_name: str, version: str | None, stage: str | None):
    """Devuelve un ModelVersion (mv) a partir de version exacta o stage (Production/Staging/latest)."""
    if version:
        mv = client.get_model_version(model_name, str(version))
        print(f"[RESOLVE] Usando versión exacta: {model_name} v{mv.version} (run_id={mv.run_id})")
        return mv

    if stage and stage.lower() != "latest":
        # Production / Staging
        vers = client.get_latest_versions(model_name, [stage])
        if not vers:
            raise RuntimeError(f"No hay versiones en stage '{stage}' para {model_name}")
        mv = vers[0]
        print(f"[RESOLVE] Usando stage {stage}: {model_name} v{mv.version} (run_id={mv.run_id})")
        return mv

    # latest => versión numérica máxima
    all_vers = client.search_model_versions(f"name='{model_name}'")
    if not all_vers:
        raise RuntimeError(f"El modelo '{model_name}' no tiene versiones.")
    mv = max(all_vers, key=lambda v: int(v.version))
    print(f"[RESOLVE] Usando latest: {model_name} v{mv.version} (run_id={mv.run_id})")
    return mv

def try_download_via_models_uri(model_name: str, spec: str, artifact_relpath: str):
    """Intenta descargar usando models:/<name>/<spec>/<artifact_relpath>"""
    model_uri = f"models:/{model_name}/{spec}"
    full_uri = f"{model_uri}/{artifact_relpath}"
    print(f"[DOWNLOAD models:/] {full_uri}")
    try:
        local_path = mlflow.artifacts.download_artifacts(artifact_uri=full_uri)
        print(f"[OK] Descargado a: {local_path}")
        # Si es JSON, intenta abrirlo
        if local_path.endswith(".json"):
            with open(local_path, "r") as f:
                j = json.load(f)
            print(f"[OK] JSON keys: {list(j)[:10]}")
        return True
    except Exception as e:
        print(f"[ERROR models:/] {short_exc(e)}")
        return False

def try_download_via_runs_uri(run_id: str, artifact_relpath: str):
    """Intenta descargar usando runs:/<run_id>/artifacts/<artifact_relpath>"""
    full_uri = f"runs:/{run_id}/artifacts/{artifact_relpath}"
    print(f"[DOWNLOAD runs:/] {full_uri}")
    try:
        local_path = mlflow.artifacts.download_artifacts(artifact_uri=full_uri)
        print(f"[OK] Descargado a: {local_path}")
        if local_path.endswith(".json"):
            with open(local_path, "r") as f:
                j = json.load(f)
            print(f"[OK] JSON keys: {list(j)[:10]}")
        return True
    except Exception as e:
        print(f"[ERROR runs:/] {short_exc(e)}")
        return False

def roundtrip_artifact_write_read(tmp_content: str = "hello-mlflow", artifact_name: str = "diag_artifact.txt"):
    """
    Crea un experimento temporal, loggea un artifact y lo lee de vuelta.
    Valida escritura/lectura del artifact store.
    """
    exp_name = f"diag_exp_{int(time.time())}"
    mlflow.set_experiment(exp_name)
    print(f"[ROUNDTRIP] Experiment temporal: {exp_name}")
    try:
        with mlflow.start_run(run_name="diag_run") as run:
            run_id = run.info.run_id
            print(f"[ROUNDTRIP] run_id: {run_id}")
            with tempfile.TemporaryDirectory() as td:
                p = os.path.join(td, artifact_name)
                with open(p, "w", encoding="utf-8") as f:
                    f.write(tmp_content)
                mlflow.log_artifact(p, artifact_path="diag_folder")
            # Descargar de vuelta
            ok = try_download_via_runs_uri(run_id, f"diag_folder/{artifact_name}")
            return ok
    except Exception as e:
        print(f"[ROUNDTRIP][ERROR] {short_exc(e)}")
        print(traceback.format_exc(limit=2))
        return False

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--tracking-uri", default=os.getenv("TRACKING_URI", ""), help="Override del TRACKING_URI")
    parser.add_argument("--model-name", help="Nombre del Registered Model")
    parser.add_argument("--version", help="Versión exacta del modelo (numérica)")
    parser.add_argument("--stage", default="latest", help="Stage (Production/Staging/latest) si no usas --version")
    parser.add_argument("--artifact-path", default="model/config.json", help="Ruta relativa dentro de artifacts")
    args = parser.parse_args()

    if args.tracking_uri:
        mlflow.set_tracking_uri(args.tracking_uri)

    tracking_uri = mlflow.get_tracking_uri()
    print(f"[INFO] TRACKING_URI: {tracking_uri}")

    # Chequeos HTTP directos (si 'requests' está instalado)
    maybe_http_check(tracking_uri)

    client = MlflowClient()

    print("\n=== LISTAR METADATA ===")
    list_metadata(client)

    if args.model_name:
        print("\n=== RESOLVER MODEL VERSION ===")
        try:
            mv = resolve_model_version(client, args.model_name, args.version, args.stage)
        except Exception as e:
            print(f"[RESOLVE][ERROR] {short_exc(e)}")
            mv = None

        if mv:
            spec = args.version if args.version else (args.stage if args.stage else "latest")

            print("\n=== DESCARGA VIA models:/ ===")
            ok_models = try_download_via_models_uri(args.model_name, str(spec), args.artifact_path)

            print("\n=== DESCARGA VIA runs:/ (fallback) ===")
            ok_runs = try_download_via_runs_uri(mv.run_id, args.artifact_path)

            if not ok_models and ok_runs:
                print("\n[SUGERENCIA] 'runs:/' funciona pero 'models:/' falla → suele ser problema de proxy o ruta de artifacts del Model Registry.")
            elif ok_models and not ok_runs:
                print("\n[NOTA] 'models:/' funciona pero 'runs:/' falla (raro). Revisa permisos del run o artifact_path.")
            elif not ok_models and not ok_runs:
                print("\n[ALERTA] Ambos fallaron. Revisa:")
                print("  - TRACKING_URI (host:puerto correctos, reverse proxy)")
                print("  - Config de artifact store (S3/MinIO/FS) y credenciales en el proceso del servidor")
                print("  - Logs del servidor MLflow / Nginx")
    else:
        print("\n[SKIP] No se pasó --model-name; omito descargas de artifacts del modelo.")

    print("\n=== ROUNDTRIP WRITE/READ ARTIFACT ===")
    ok_round = roundtrip_artifact_write_read()
    if ok_round:
        print("[ROUNDTRIP] OK — el servidor pudo escribir y leer artifacts.")
    else:
        print("[ROUNDTRIP] ERROR — falla en escritura/lectura de artifacts.")

    print("\nListo. Revisa arriba los bloques [ERROR] / [HTTP] para pistas concretas.")

if __name__ == "__main__":
    main()
